# 第8章 文埋め込み

## 8.3 文埋め込みモデルの実装

### 8.3.2 教師ありのSimCSEの実装

SimCSE (Simple Contrastive learning of Sentence Embedding)

Contrastive learning: 対照学習

#### 準備

In [1]:
# 必要なパッケージをインストールする
!pip install 'datasets<4.0.0' scipy 'transformers[ja,torch]<4.41.0'

In [2]:
!pip install "peft<0.12"

In [3]:
from transformers.trainer_utils import set_seed

set_seed(42)

#### データセットの読み込みと前処理

In [4]:
from datasets import load_dataset

# Hugging Face Hubのllm-book/jsnliのリポジトリから
# JSNLIの訓練セットを読み込む
jsnli_dataset = load_dataset("llm-book/jsnli", split="train")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


In [5]:
# 訓練セットの形式と事例数を確認する
print(jsnli_dataset)

Dataset({
    features: ['premise', 'hypothesis', 'label'],
    num_rows: 533005
})


In [6]:
from pprint import pprint

# JSNLIの訓練セットの内容を確認する
pprint(jsnli_dataset[0])
pprint(jsnli_dataset[1])

{'hypothesis': '男 は 魔法 の ショー の ため に ナイフ を 投げる 行為 を 練習 して い ます 。',
 'label': 'neutral',
 'premise': 'ガレージ で 、 壁 に ナイフ を 投げる 男 。'}
{'hypothesis': '女性 が 畑 で 踊って い ます 。',
 'label': 'contradiction',
 'premise': '茶色 の ドレス を 着た 女性 が ベンチ に 座って い ます 。'}


In [7]:
import csv
import random
from typing import Iterator

# JSNLIの訓練セットから、前提文とラベルごとに仮説分をまとめたdictを作成する
premise2hypotheses = {}

premises = jsnli_dataset["premise"]  # 前提文
hypotheses = jsnli_dataset["hypothesis"]  # 仮説文
labels = jsnli_dataset["label"]  # ラベル

for premise, hypothesis, label in zip(premises, hypotheses, labels):
    if premise not in premise2hypotheses:
        premise2hypotheses[premise] = {
            "entailment": [],
            "neutral": [],
            "contradiction": [],
        }

    premise2hypotheses[premise][label].append(hypothesis)

In [8]:

from datasets import Dataset

def generate_sup_train_example() -> Iterator[dict[str, str]]:
    """教師ありSimCSEの訓練セットの事例を生成"""
    # JSNLIのデータから（前提文、「含意」ラベルの仮説文、「矛盾」ラベルの仮説文）の3つ組を生成する
    for premise, hypotheses in premise2hypotheses.items():
        # 「矛盾」ラベルの仮説文が一つもない場合はスキップする
        if len(hypotheses["contradiction"]) == 0:
            continue

        # 「含意」ラベルの仮説文一つにつき、「矛盾」ラベルの仮説文一つをランダムに関連付ける
        for entailment_hypothesis in hypotheses["entailment"]:
            contradiction_hypothsis = random.choice(
                hypotheses["contradiction"]
            )
            # （前提文、「含意」ラベルの仮説文、「矛盾」ラベルの仮説文）の3つ組をdictとして生成
            yield {
                "premise": premise,
                "entailment_hypothesis": entailment_hypothesis,
                "contradiction_hypothesis": contradiction_hypothsis,
            }

# 定義したジェネレータ関数を用いて、教師ありSimCSEの訓練セットを構築する
sup_train_dataset = Dataset.from_generator(generate_sup_train_example)

In [9]:
# 訓練セットの形式と事例数を確認する
print(sup_train_dataset)

Dataset({
    features: ['premise', 'entailment_hypothesis', 'contradiction_hypothesis'],
    num_rows: 173438
})


In [10]:
# 訓練セットの内容を確認する
pprint(sup_train_dataset[0])
pprint(sup_train_dataset[1])

{'contradiction_hypothesis': '男 が 台所 の テーブル で 本 を 読んで い ます 。',
 'entailment_hypothesis': 'ガレージ に 男 が い ます 。',
 'premise': 'ガレージ で 、 壁 に ナイフ を 投げる 男 。'}
{'contradiction_hypothesis': '黒人 は デスクトップ コンピューター を 使用 し ます 。',
 'entailment_hypothesis': '人 は 椅子 に 座って い ます 。',
 'premise': 'ラップ トップ コンピューター を 使用 して 机 に 座って いる 若い 白人 男 。'}


#### トークナイザの準備

In [11]:
from transformers import AutoTokenizer

# Hugging Face Hubにおけるモデル名を指定する
base_model_name = "cl-tohoku/bert-base-japanese-v3"
# モデル名からトークナイザを初期化する
tokenizer = AutoTokenizer.from_pretrained(base_model_name)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


#### collate関数の準備

In [12]:
import torch
from torch import Tensor
from transformers import BatchEncoding

def sup_train_collate_fn(
    examples: list[dict]
) -> dict[str, BatchEncoding | Tensor]:
    """訓練セットのミニバッチを作成"""
    premises = []
    hypotheses = []

    for example in examples:
        premises.append(example["premise"])

        entailment_hypothesis = example["entailment_hypothesis"]
        contradiction_hypothesis = example["contradiction_hypothesis"]

        hypotheses.extend(
            [entailment_hypothesis, contradiction_hypothesis]
        )
    
    # ミニバッチに含まれる前提文と仮説文にトークナイザを適用する
    tokenized_premises = tokenizer(
        premises,
        padding=True,
        truncation=True,
        max_length=32,
        return_tensors="pt",
    )
    tokenized_hypotheses = tokenizer(
        hypotheses,
        padding=True,
        truncation=True,
        max_length=32,
        return_tensors="pt",
    )

    # 前提文とか節分の類似度行列における正例ペアの位置を示すTensorを作成する
    # 行列のi行目の事例（前提文）に対して2*i行目の要素（仮説文）が正例ペアとなる
    labels = torch.arange(0, 2*len(premises), 2)

    return {
        "tokenized_texts_1": tokenized_premises,
        "tokenized_texts_2": tokenized_hypotheses,
        "labels": labels,
    }

In [13]:
def eval_collate_fn(
    examples: list[dict],
) -> dict[str, BatchEncoding | Tensor]:
    """SimCSEの検証・テストセットのミニバッチを作成"""
    # ミニバッチの文ペアに含まれる文（文1と文2）のそれぞれに
    # トークナイザを適用する
    tokenized_texts_1 = tokenizer(
        [example["sentence1"] for example in examples],
        padding=True,
        truncation=True,
        max_length=512,
        return_tensors="pt",
    )
    tokenized_texts_2 = tokenizer(
        [example["sentence2"] for example in examples],
        padding=True,
        truncation=True,
        max_length=512,
        return_tensors="pt",
    )

    # 文1と文2の類似度行列における正例ペアの位置を示すTensorを作成する
    # 行列のi行目の事例（文1）に対して
    # i列目の事例（文2）との組が正例ペアとなる
    labels = torch.arange(len(examples))

    # データセットに付与された類似度スコアのTensorを作成する
    label_scores = torch.tensor(
        [example["label"] for example in examples]
    )

    return {
        "tokenized_texts_1": tokenized_texts_1,
        "tokenized_texts_2": tokenized_texts_2,
        "labels": labels,
        "label_scores": label_scores,
    }

#### モデルの準備

In [14]:
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModel
from transformers.utils import ModelOutput

class SimCSEModel(nn.Module):
    """SimCSEのモデル"""

    def __init__(
        self,
        base_model_name: str,
        mlp_only_train: bool = False,  # MLP: 多層パーセプトロン
        temperture: float = 0.5
    ):
        """モデルの初期化"""
        super().__init__()

        # モデル名からエンコーダを初期化する
        self.encoder = AutoModel.from_pretrained(base_model_name)
        # パラメータをメモリ上に隣接した形で配置
        # これを実行しない場合、モデルの保存でエラーになることがある
        for param in self.encoder.parameters():
            param.data = param.data.contiguous()

        # MLPの次元数
        self.hidden_size = self.encoder.config.hidden_size
        # MLP層の線形層
        self.dense = nn.Linear(self.hidden_size, self.hidden_size)
        # MLP層の活性化関数
        self.activation = nn.Tanh()

        # MLP層による変換を訓練時にのみ適用するよう設定するフラグ
        self.mlp_only_train = mlp_only_train
        # 交差エントロピー損失の計算時に使用する温度
        self.temperture = temperture

    def encode_texts(self, tokenized_texts: BatchEncoding) -> Tensor:
        """エンコーダを用いて文をベクトルに変換"""
        # トークナイズされた文をエンコーダに入力する
        encoded_texts = self.encoder(**tokenized_texts)
        # モデルの最終層の出力(last_hidden_state)の
        # [CLS]トークン(0番目の位置のトークン)のベクトルを取り出す
        encoded_texts = encoded_texts.last_hidden_state[:,0]

        # self.mlp_only_trainのフラグがTrueに設定されていて、
        # かつ訓練時でない場合、MLP層の変換を適用せずにベクトルを返す
        if self.mlp_only_train and not self.training:
            return encoded_texts

        # MLP層におけるベクトルの変換を行う
        encoded_texts = self.dense(encoded_texts)
        encoded_texts = self.activation(encoded_texts)

        return encoded_texts

    def forward(
        self,
        tokenized_texts_1: BatchEncoding,
        tokenized_texts_2: BatchEncoding,
        labels: Tensor,
        label_scores: Tensor | None = None
    ) -> ModelOutput:
        """モデルの前向き計算を定義"""
        # 文ペアをベクトルに変換する
        encoded_texts_1 = self.encode_texts(tokenized_texts_1)
        encoded_texts_2 = self.encode_texts(tokenized_texts_2)

        # 文ペアの類似度行列を作成する
        sim_matrix = F.cosine_similarity(
            encoded_texts_1.unsqueeze(1),
            encoded_texts_2.unsqueeze(0),
            dim=2
        )

        # 交差エントロピー損失を求める
        loss = F.cross_entropy(sim_matrix / self.temperture, labels)

        # 性能評価に使用するため、正例ペアに対するスコアを類似度行列から取り出す
        positive_mask = F.one_hot(labels, sim_matrix.size(1)).bool()
        positive_scores = torch.masked_select(
            sim_matrix, positive_mask
        )

        return ModelOutput(loss=loss, scores=positive_scores)

In [15]:
# 教師ありSimCSEのモデルを初期化する
sup_model = SimCSEModel(base_model_name, mlp_only_train=False)

#### 検証セット、テストセットの準備

In [16]:
# Hugging Face Hubのllm-book/JGLUEのリポジトリから
# JSTSデータセットの訓練セットと検証セットを読み込み、
# それぞれをSimCSEの検証セットとテストセットとして使用する
# JSTSはテストセットが公開されていないため、
# 検証セット -> "train", テストセット -> "validation"で対応
valid_dataset = load_dataset(
    "llm-book/JGLUE", name="JSTS", split="train"
)
test_dataset = load_dataset(
    "llm-book/JGLUE", name="JSTS", split="validation"
)

#### `Trainer`の準備

In [17]:
from scipy.stats import spearmanr
from transformers import EvalPrediction

def compute_metrics(p: EvalPrediction) -> dict[str, float]:
    """モデルが予測したスコアと評価用データのスコアのスピアマンの順位相関係数を計算"""
    scores = p.predictions
    labels, label_scores = p.label_ids

    spearman = spearmanr(scores, label_scores).statistic

    return {"spearman": spearman}

In [18]:
from transformers import TrainingArguments

# 教師なしSimCSEの訓練のハイパーパラメータを設定する
sup_training_args = TrainingArguments(
    output_dir="outputs_sup_simcse",    # 結果の保存先フォルダ
    per_device_train_batch_size=128,    # 訓練時のバッチサイズ
    per_device_eval_batch_size=128,     # 評価時のバッチサイズ
    learning_rate=5e-5,                 # 学習率
    num_train_epochs=3,                 # 訓練エポック数
    evaluation_strategy="steps",        # 検証セットによる評価のタイミング
    eval_steps=250,                     # 検証セットによる評価を行う訓練ステップ数の間隔
    logging_steps=250,                  # ロギングを行う訓練ステップ数の間隔
    save_steps=250,                     # チェックポイントを保存する訓練ステップ数の間隔
    save_total_limit=1,                 # 保存するチェックポイントの最大数
    fp16=True,                          # 自動混合精度演算の有効化
    load_best_model_at_end=True,        # 最良のモデルを訓練終了後に読み込むか
    metric_for_best_model="spearman",   # 最良のモデルを決定する評価指標
    remove_unused_columns=False,        # データセットの不要フィールドを削除するか
    report_to="none",                   # 外部ツールへのログを無効化
)

In [19]:
from datasets import Dataset
from torch.utils.data import DataLoader
from transformers import Trainer

class SimCSETrainer(Trainer):
    """SimCSEの訓練に使用するTrainer"""

    def get_eval_dataloader(
        self, eval_dataset: Dataset | None = None
    ):
        """
        検証・テストセットのDataLoaderでeval_collate_fnを使うように
        Trainerのget_eval_dataloaderをオーバーライド
        """
        if eval_dataset is None:
            eval_dataset = self.eval_dataset

        return DataLoader(
            eval_dataset,
            batch_size=64,
            collate_fn=eval_collate_fn,
            pin_memory=True,
        )

# 教師なしSimCSEのTrainerを初期化する
sup_trainer = SimCSETrainer(
    model=sup_model,
    args=sup_training_args,
    data_collator=sup_train_collate_fn,
    train_dataset=sup_train_dataset,
    eval_dataset=valid_dataset,
    compute_metrics=compute_metrics,
)

#### 訓練の実行

In [20]:
# 教師なしSimCSEの訓練を行う
sup_trainer.train()

Step,Training Loss,Validation Loss,Spearman
250,4.247900,3.101734,0.672901
500,4.117700,3.094665,0.686231
750,4.091100,3.079015,0.689937
1000,4.073300,3.102636,0.688719
1250,4.058000,3.094247,0.694863
1500,4.034600,3.098428,0.694161
1750,4.020000,3.103362,0.688974
2000,4.013600,3.084401,0.695005
2250,4.012600,3.095391,0.693542
2500,4.006500,3.099431,0.693412


TrainOutput(global_step=4065, training_loss=4.03271579718971, metrics={'train_runtime': 1593.8069, 'train_samples_per_second': 326.46, 'train_steps_per_second': 2.55, 'total_flos': 0.0, 'train_loss': 4.03271579718971, 'epoch': 3.0})

#### 性能評価

In [21]:
# 検証セットで教師なしSimCSEのモデルの評価を行う
sup_trainer.evaluate(valid_dataset)

{'eval_loss': 3.0844004154205322,
 'eval_spearman': 0.6950051500791299,
 'eval_runtime': 9.0712,
 'eval_samples_per_second': 1372.585,
 'eval_steps_per_second': 10.803,
 'epoch': 3.0}

In [22]:
# テストセットで教師なしSimCSEのモデルの評価をおこなう
sup_trainer.evaluate(test_dataset)

{'eval_loss': 3.085846424102783,
 'eval_spearman': 0.7036075258152633,
 'eval_runtime': 1.1134,
 'eval_samples_per_second': 1308.654,
 'eval_steps_per_second': 10.778,
 'epoch': 3.0}

#### トークナイザとモデルの保存

#### Google Driveへの保存

In [23]:
from google.colab import drive
drive.mount("drive")



# 保存されたモデルをGoogleドライブのフォルダにコピーする
!cp -r outputs_sup_simcse drive/MyDrive/llm-book

Mounted at drive
